In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import yfinance as yf


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.final_model import TransitionPredictor
from src.features import add_all_features, add_target_ma_cross, add_distances_GC
from src.data_loader import load_data

In [2]:
def add_target(df: pd.DataFrame, period=30):
    df1 = df.copy()
    df1['Golden_Cross'] = df1['Golden_Cross'].replace({'Non-Bullish': 0, 'Bullish': 1})
    

    df1['Transition'] = 0
    

    label = df1['Golden_Cross'].iloc[0]
    indices = []
    for i in range(1, len(df1)):
        if df1['Golden_Cross'].iloc[i] != label:
            label = df1['Golden_Cross'].iloc[i]
            indices.append(i)

    for i in indices:
        start = max(0, i - period)  
        df1.loc[start:i-1, 'Transition'] = 1  
    
    return df1, indices


def preprocess_asset(ticker):

    df_ticker = load_data(ticker=ticker)
    
    df_ticker = add_all_features(df_ticker)
    df_ticker = add_target_ma_cross(df_ticker)
    df_ticker = add_distances_GC(df_ticker)
    df_ticker = add_target(df_ticker)[0]

    # Additional features created on the models notebook
    df_final = df_ticker.copy()
    
    df_final['MA50'] = df_final['Close'].rolling(window=50).mean()
    df_final['MA200'] = df_final['Close'].rolling(window=200).mean()
    df_final['MA_velocity'] = (df_final['MA50'] - df_final['MA50'].shift(5)) - (df_final['MA200'] - df_final['MA200'].shift(5))
    df_final['MA50_slope'] = df_final['MA50'].diff(10) / df_final['MA50']
    df_final['Distance_normalized'] = df_final['Distance_GC'] / df_final['Volatility'].rolling(50).mean()
    df_final['MA50_accel'] = df_final['MA50'].diff(5) - df_final['MA50'].diff(10)
    df_final['MA_cross_momentum'] = df_final['MA50_accel'] / df_final['Distance_GC'].abs()

    # Add VIX spike
    vix = yf.Ticker("^VIX")
    vix_data = vix.history(start="2000-01-01", end="2024-12-31")
    vix_close = vix_data['Close'].to_frame()
    vix_close.columns = ['VIX']
    vix_close.index = vix_close.index.tz_localize(None)  # Remove timezone

    df_vix = df_final.copy()
    
    df_vix['Date'] = pd.to_datetime(df_vix['Date']).dt.tz_localize(None)
    
    # Set index et merge
    df_indexed = df_vix.set_index('Date')
    df_with_vix = pd.concat([df_indexed, vix_close], axis=1)
    df_with_vix = df_with_vix.reset_index()

    print(f"Missing VIX: {df_with_vix['VIX'].isna().sum()}")

    df_with_vix['VIX_spike'] = df_with_vix['VIX'] / df_with_vix['VIX'].rolling(60).mean()

    df_with_vix = df_with_vix.drop(['Close', 'High', 'Low', 'Open', 'Golden_Cross', 'Date',
                                    'MA50', 'MA200', 'VIX', 'MA50_accel', 'Dividends',
                                     'Stock Splits', 'Capital Gains', 'Volume', 'Log Return',
                                      'Stoch_K'], axis=1)

    return df_with_vix

#Test w SPY

spy_test = preprocess_asset('SPY')
print(spy_test.columns.tolist())
spy_test.head(500)



Missing VIX: 0
['Return', 'Volatility', 'Cumulated_Return_5d', 'RSI14', 'Volume_ROC', 'ATR', 'Distance_GC', 'Transition', 'MA_velocity', 'MA50_slope', 'Distance_normalized', 'MA_cross_momentum', 'VIX_spike']


,Return,Volatility,Cumulated_Return_5d,RSI14,Volume_ROC,ATR,Distance_GC,Transition,MA_velocity,MA50_slope,Distance_normalized,MA_cross_momentum,VIX_spike
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
1,-0.039106,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
2,0.001789,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
3,-0.016072,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
4,0.058077,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,-0.001914,0.011315,0.007215,49.387367,-66.770109,0.990108,-0.035864,0,0.590002,0.012723,-2.928428,-12.916578,0.799689
496,0.005491,0.011344,0.006756,43.163715,-59.114426,0.937577,-0.034343,0,0.571538,0.012101,-2.823337,-12.953662,0.792707
497,0.006068,0.010551,0.005779,46.521404,-47.658419,0.929447,-0.032841,0,0.534223,0.011692,-2.721905,-13.987953,0.772202
498,-0.000517,0.010164,0.015256,49.325866,-43.822715,0.892406,-0.031100,0,0.558892,0.012388,-2.600824,-15.440455,0.769969


In [3]:
column_order = ['Return', 'Volatility', 'Cumulated_Return_5d', 'RSI14', 'Volume_ROC', 'ATR', 'Transition', 'VIX_spike', 'Distance_GC', 'MA_velocity', 'MA50_slope', 'Distance_normalized', 'MA_cross_momentum']
spy_test = spy_test[column_order]

In [4]:
X = spy_test.drop(['Transition'], axis=1)
y = spy_test['Transition']


X_train = X.iloc[:5030]
X_test = X.iloc[5030:]
y_train = y.iloc[:5030]
y_test = y.iloc[5030:]

print(f"Train: {len(X_train)} samples, {y_train.sum()} positives")
print(f"Test: {len(X_test)} samples, {y_test.sum()} positives")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Train: 5030 samples, 593 positives
Test: 1258 samples, 120 positives
X_train shape: (5030, 12)
X_test shape: (1258, 12)


In [5]:
predictor = TransitionPredictor(threshold=0.65)

predictor.fit(X_train, y_train)

print("\nTest set evaluation:")
metrics = predictor.evaluate(X_test, y_test, verbose=True)

print("\nFeature Importances:")
print(predictor.get_feature_importance())


Test set evaluation:

📊 ÉVALUATION (threshold=0.65)
Precision: 0.586
Recall:    0.708
F1 Score:  0.642

Classification Report:
               precision    recall  f1-score   support

No Transition       0.97      0.95      0.96      1138
   Transition       0.59      0.71      0.64       120

     accuracy                           0.92      1258
    macro avg       0.78      0.83      0.80      1258
 weighted avg       0.93      0.92      0.93      1258


Feature Importances:
                feature  importance
10  Distance_normalized    0.321306
7           Distance_GC    0.298400
9            MA50_slope    0.102003
1            Volatility    0.088979
11    MA_cross_momentum    0.071016
8           MA_velocity    0.045646
6             VIX_spike    0.035975
5                   ATR    0.018686
3                 RSI14    0.008661
2   Cumulated_Return_5d    0.004646
4            Volume_ROC    0.003131
0                Return    0.001550


In [11]:
def validate_cross_assets(tickers: list):
    results = []
    
    # Train on SPY
    print("="*70)
    print("Training on SPY...")
    print("="*70)
    
    spy_df = preprocess_asset('SPY')
    column_order = ['Return', 'Volatility', 'Cumulated_Return_5d', 'RSI14', 
                   'Volume_ROC', 'ATR', 'Transition', 'VIX_spike', 'Distance_GC', 
                   'MA_velocity', 'MA50_slope', 'Distance_normalized', 'MA_cross_momentum']
    spy_df = spy_df[column_order]
    
    X_spy = spy_df.drop(['Transition'], axis=1)
    y_spy = spy_df['Transition']
    X_train_spy = X_spy.iloc[:5030]
    y_train_spy = y_spy.iloc[:5030]
    
    # DEBUG
    print(f"SPY - y_spy NaN: {y_spy.isna().sum()}")
    print(f"SPY - X_spy NaN: {X_spy.isna().sum().sum()}")
    
    predictor = TransitionPredictor(threshold=0.65)
    predictor.fit(X_train_spy, y_train_spy)
    
    print(f"\nModel trained on SPY")
    
    # Test on all assets
    for ticker in tickers:
        print(f"\n{'#'*70}")
        print(f"# Testing on {ticker}")
        print(f"{'#'*70}")
        
        df = preprocess_asset(ticker)
        df = df[column_order]
        
        X = df.drop(['Transition'], axis=1)
        y = df['Transition']
        
        X_test = X.iloc[5030:]
        y_test = y.iloc[5030:]
        
        print(f"\n{ticker} - Test set size: {len(y_test)}")
        print(f"{ticker} - Test positives: {y_test.sum()} ({y_test.mean()*100:.1f}%)")
        
        metrics = predictor.evaluate(X_test, y_test, verbose=True)
        
        results.append({
            'Asset': ticker,
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1': metrics['f1'],
            'N_samples': len(y_test),
            'N_positive': y_test.sum(),
            'Pct_positive': y_test.mean() * 100
        })
    
    # Summary
    results_df = pd.DataFrame(results)
    print("\n" + "="*70)
    print("CROSS-ASSET VALIDATION SUMMARY")
    print("="*70)
    print(results_df.to_string(index=False))
    
    return results_df


# Run validation
tickers = ['SPY', 'DIA', 'QQQ']
results = validate_cross_assets(tickers)

Training on SPY...
Missing VIX: 0
SPY - y_spy NaN: 0
SPY - X_spy NaN: 986

Model trained on SPY

######################################################################
# Testing on SPY
######################################################################
Missing VIX: 0

SPY - Test set size: 1258
SPY - Test positives: 120 (9.5%)

📊 ÉVALUATION (threshold=0.65)
Precision: 0.586
Recall:    0.708
F1 Score:  0.642

Classification Report:
               precision    recall  f1-score   support

No Transition       0.97      0.95      0.96      1138
   Transition       0.59      0.71      0.64       120

     accuracy                           0.92      1258
    macro avg       0.78      0.83      0.80      1258
 weighted avg       0.93      0.92      0.93      1258


######################################################################
# Testing on DIA
######################################################################
Missing VIX: 0

DIA - Test set size: 1258
DIA - Test positives: 120 (9

In [7]:
# SPY
spy = preprocess_asset('SPY')
print(f"SPY - Total NaN in target: {spy['Transition'].isna().sum()}")
print(f"SPY - Shape: {spy.shape}")

# DIA
dia = preprocess_asset('DIA')
print(f"\nDIA - Total NaN in target: {dia['Transition'].isna().sum()}")
print(f"DIA - Shape: {dia.shape}")

# Compare
print(f"\nDifférence de NaN: {dia['Transition'].isna().sum() - spy['Transition'].isna().sum()}")

Missing VIX: 0
SPY - Total NaN in target: 0
SPY - Shape: (6288, 13)
Missing VIX: 0

DIA - Total NaN in target: 0
DIA - Shape: (6288, 13)

Différence de NaN: 0
